In [27]:
### IMAGE INVENTORY
# create an .xlsx file with one sheet per label listing all images
# Camera_ID | SD_ID | Deploy | Img_ID | path
# 
# uses a file with all the image diretories and correcsponding label sheets
# label_sheet_path | label_sheet_name | image_dir

# doesnt work because of how images are labeled
#   in the sheet they are labeled 1,2,...,998,999,1001,1002
#   but images are put in folders of 999 images called 100MEDIA, 101MEDIA,...
# solution (v2): get path directly from image name with a function instead of finding image in subdirs

import pandas as pd
import re
from pathlib import Path


def extract_values(input_str):
    match = re.search(r'\(\s*(\w+)\s*,\s*(\w+)\s*\)', input_str)
    cam_id, sd_id = (match.group(1), match.group(2)) if match else (None, None)
    deploy_match = re.search(r'(\d+)\s\(', input_str, re.IGNORECASE)
    deploy_id = deploy_match.group(1) if deploy_match else None
    return cam_id, sd_id, deploy_id

def createImageInventory():
    # set up data structure
    label_buckets = {i: pd.DataFrame(columns=['Camera_ID', 'SD_ID', 'Deploy', 'Image_ID', 'Image_Path']) for i in range(1, 7)}
    # load data inventory and loop thru it
    inventory_path = 'data_inventory_with.xlsx'
    sheet_name = 'usable-data'
    inventory = pd.read_excel(inventory_path, sheet_name=sheet_name)

    for inv_index, inv_entry in inventory.iterrows():
        # debug
        print(f"##### -------------------- LOOP {inv_index} -------------------- #####")
        # get cam&sd ids & deploy
        id1, id2, id3 = extract_values(inv_entry['data_sheet_name'])
        # load label sheet
        label_sheet = pd.read_excel(
            inv_entry['data_sheet_path'],
            sheet_name=inv_entry['data_sheet_name'],
            header=1 
        )
        # loop thru rows
        for row_number, row in label_sheet.iterrows():
            # get image ID and name
            img_id = (
                row.get('picture number camera output ') or
                row.get('Picture number camera output ') or
                row.get('Picture Number camera output ') or
                row.get('picture number camera output') or
                row.get('Picture number camera output') or
                row.get('Picture Number camera output')
            )
            if img_id is None: 
                print(f"no image label row found in {id1}:{id2}:{id3}/{row_number}  ### this breaks the loop and goes to the next inventory entry")
                print(row.iloc[26])
                break#continue
            img_name = f'DSCF{img_id:04d}.JPG'
            # search for correct image file and get the path
            img_found = list(Path(inv_entry['folder_path']).rglob(img_name))
            if not img_found:
                print(f"File '{img_name}' not found in '{inv_entry['folder_path']}' ### this breaks the loop and goes to the next inventory entry")
                break#continue
            img_path = str(img_found[0].resolve())
            # get label
            img_label = row['Water Stream Rating']
            # append it all to the right bucket
            img_data = {
                'Camera_ID': id1, 
                'SD_ID': id2,
                'Deploy': id3,
                'Image_ID': img_id,
                'Image_Path': img_path
            }
            if img_label not in label_buckets:
                print(f"Invalid label for Image #{img_id} in {id1}:{id2}:{id3} ### this breaks the loop and goes to the next inventory entry")
                break
            label_buckets[img_label] = pd.concat(
                [label_buckets[img_label], pd.DataFrame([img_data])],
                ignore_index = True
            )
            
    # save as excel    
    with pd.ExcelWriter('image_inventory.xlsx', engine='openpyxl') as writer:
        for i in range(1, 7):
            label_buckets[i].to_excel(writer, sheet_name=f'Label_{i}', index=False)

    return

createImageInventory()

##### -------------------- LOOP 0 -------------------- #####
##### -------------------- LOOP 1 -------------------- #####
File 'DSCF1001.JPG' not found in 'E:\backup\Stream Camera Project\2020_StreamCams\Photos\31-Jan Deploy 4 (1000, M1)' ### this breaks the loop and goes to the next inventory entry
##### -------------------- LOOP 2 -------------------- #####
File 'DSCF1001.JPG' not found in 'E:\backup\Stream Camera Project\2019_StreamCams\2019_Photos\Deployment 2 (1000, M10)' ### this breaks the loop and goes to the next inventory entry
##### -------------------- LOOP 3 -------------------- #####
File 'DSCF1001.JPG' not found in 'E:\backup\Stream Camera Project\2020_StreamCams\Photos\18-Mar Deploy 5 (1000, M11)' ### this breaks the loop and goes to the next inventory entry
##### -------------------- LOOP 4 -------------------- #####
File 'DSCF1001.JPG' not found in 'E:\backup\Stream Camera Project\2020_StreamCams\Photos\2-Jul Deploy 7 (1000, M2)' ### this breaks the loop and goes to t

ValueError: Unknown format code 'd' for object of type 'float'